# Bloco C - Modelos de Machine Learning

## 1. Carregar a amostra rotulada

In [0]:
pdf = spark.table("workspace.senhas.senhas_rotuladas").toPandas()
print(f"Registros: {len(pdf):,}")

FEATURES_CLUSTER = [
    "tamanho", "tem_maiuscula", "tem_minuscula", "tem_numero", "tem_especial",
    "qtd_maiusculas", "qtd_minusculas", "qtd_numeros", "qtd_especiais"
]

Registros: 500,000


## 2. Preparar os dados para treino

Separação treino/teste e normalização, para cada cenário de rotulagem (K-Means e Regras Manuais).

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cenario K-Means
X_km = pdf[FEATURES_CLUSTER]
y_km = pdf["forca_kmeans"]
X_train_km, X_test_km, y_train_km, y_test_km = train_test_split(
    X_km, y_km, test_size=0.2, random_state=42)
scaler_km = StandardScaler()
X_train_km_s = scaler_km.fit_transform(X_train_km)
X_test_km_s = scaler_km.transform(X_test_km)

# Cenario Regras Manuais
X_rg = pdf[FEATURES_CLUSTER]
y_rg = pdf["forca_regras"]
X_train_rg, X_test_rg, y_train_rg, y_test_rg = train_test_split(
    X_rg, y_rg, test_size=0.2, random_state=42)
scaler_rg = StandardScaler()
X_train_rg_s = scaler_rg.fit_transform(X_train_rg)
X_test_rg_s = scaler_rg.transform(X_test_rg)

print("Dados preparados para os dois cenarios.")

Dados preparados para os dois cenarios.


## 3. Definir os modelos

In [0]:
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

modelos = {
    "Rede Neural (MLP)"   : MLPClassifier(hidden_layer_sizes=(5,), max_iter=500, random_state=42),
    "Random Forest"       : RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42),
    "Regressao Logistica" : LogisticRegression(max_iter=1000, random_state=42)
}

## 4. Treinar e avaliar

Cada modelo é treinado nos dois cenários, comparando a acurácia.

In [0]:
from sklearn.metrics import accuracy_score

resultados = {}
for nome, modelo in modelos.items():
    modelo.fit(X_train_km_s, y_train_km)
    acc_km = accuracy_score(y_test_km, modelo.predict(X_test_km_s))

    modelo.fit(X_train_rg_s, y_train_rg)
    acc_rg = accuracy_score(y_test_rg, modelo.predict(X_test_rg_s))

    resultados[nome] = {"K-Means": acc_km, "Regras": acc_rg}
    print(f"{nome}")
    print(f"   K-Means : {acc_km*100:.2f}%")
    print(f"   Regras  : {acc_rg*100:.2f}%\n")

Rede Neural (MLP)
   K-Means : 100.00%
   Regras  : 100.00%

Random Forest
   K-Means : 100.00%
   Regras  : 99.52%

Regressao Logistica
   K-Means : 100.00%
   Regras  : 98.40%



## 5. Relatório de classificação (cenário Regras Manuais)

In [0]:
from sklearn.metrics import classification_report

for nome, modelo in modelos.items():
    modelo.fit(X_train_rg_s, y_train_rg)
    pred = modelo.predict(X_test_rg_s)
    print(f"=== {nome} ===")
    print(classification_report(y_test_rg, pred,
          target_names=["fraca","forte","moderada"]))

=== Rede Neural (MLP) ===
              precision    recall  f1-score   support

       fraca       1.00      1.00      1.00     15869
       forte       1.00      1.00      1.00      2079
    moderada       1.00      1.00      1.00     82052

    accuracy                           1.00    100000
   macro avg       1.00      1.00      1.00    100000
weighted avg       1.00      1.00      1.00    100000

=== Random Forest ===
              precision    recall  f1-score   support

       fraca       1.00      0.97      0.98     15869
       forte       1.00      1.00      1.00      2079
    moderada       0.99      1.00      1.00     82052

    accuracy                           1.00    100000
   macro avg       1.00      0.99      0.99    100000
weighted avg       1.00      1.00      1.00    100000

=== Regressao Logistica ===
              precision    recall  f1-score   support

       fraca       0.94      0.96      0.95     15869
       forte       1.00      1.00      1.00      2079

## 6. Análise crítica — Label Leakage

Os modelos atingem acurácia próxima de 100%, especialmente no cenário K-Means. Isso **não indica um modelo excepcional**, e sim **vazamento de rótulo (label leakage)**: as features usadas para treinar os modelos são as mesmas que definiram os rótulos (tanto nas regras quanto na clusterização), tornando a predição trivial.

Documentar essa limitação é parte central do projeto — reconhecer quando uma métrica "boa demais" sinaliza um problema de desenho experimental, e não um sucesso real.